In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
import pickle
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
import joblib
from sklearn.pipeline import Pipeline

In [ ]:
def get_year(datetime_value):
  year = datetime_value.year
  return year
def get_month(datetime_value):
  month = datetime_value.month
  return month
def get_day(datetime_value):
  day = datetime_value.day
  return day
def get_hour(datetime_value):
  hour = datetime_value.hour
  return hour


In [ ]:
unpack_file=open("/content/appml-assignment1-dataset-v2.pkl","rb")
# print(type(pickle.load(unpack_file)))
data=pickle.load(unpack_file)
df_x=data["X"]
# print(df_x.head(1))
# print(df_y.head(1))
# for i in df_x.columns:
#   print(i)
df_y=data["y"]
# print(df_x.info())
# print(df_y.info())

CAD = df_x[['date', 'CAD-open', 'CAD-high', 'CAD-low', 'CAD-close']]
CAD_selected = df_x[['CAD-open', 'CAD-high', 'CAD-low', 'CAD-close']]
# print(CAD.info())
# print(CAD_selected.info())
CAD['year']=CAD["date"].apply(get_year)
CAD['month']=CAD["date"].apply(get_month)
CAD['day']=CAD["date"].apply(get_day)
CAD['hour']=CAD["date"].apply(get_hour)
CAD.describe()
# print(CAD.head(3))
# print(df_y.head(3))
# CAD_selected.head(3)



In [ ]:
sns.heatmap(df_x.isnull(), cmap='viridis', cbar=False)
# df_x.dropna()

In [ ]:
CAD_year=CAD.groupby("year")
CAD_day=CAD.groupby("day")
CAD_month=CAD.groupby("month")
CAD_hour=CAD.groupby("hour")
CAD_day.mean()
# CAD_month.min()
# CAD_hour.max()
# CAD_year.count()
# sns.heatmap(CAD_year.count(),cmap="rainbow")

In [ ]:
sns.distplot(CAD_selected["CAD-low"],kde=False)

In [ ]:
sns.distplot(CAD["year"],kde=False)
#The year 2020 contains the least amount of data

In [ ]:
# sns.barplot(data=CAD,x=CAD["year"],y="CAD-close")
# sns.barplot(data=CAD,x=CAD["year"],y="CAD-high")
# sns.countplot(data=CAD,x=CAD["year"])

In [ ]:
sns.jointplot(data=CAD,x="date",y="CAD-open",kind="scatter")
#It is clear in this digram that in 2016 the price the CAD began the hour was higher than all years

In [ ]:
sns.jointplot(data=CAD,x="CAD-high",y="CAD-close",kind="hex")
#there is a positive correlation between CAD-high and CAD-close becase the straight line and There is a large concentration of data around the number 1.3

In [ ]:
# sns.jointplot(data=CAD,x="CAD-high",y=data["y"],kind="reg")

In [ ]:
# sns.boxplot(data=CAD,x="year",y="CAD-high",palette="rainbow",hue="month")
# sns.boxplot(data=CAD,x="year",y="CAD-high",palette="rainbow")

In [ ]:
# sns.histplot(df_y,kde=True)
sns.heatmap(CAD.corr(),annot=True)
#The higher the number, the greater the relationship between the two elements. In the case of a negative number, the relationship is inverse

In [ ]:
sns.pairplot(data=CAD_selected)#the most important one to get the relations between the data

In [ ]:
# null_value=np.array(CAD.isnull().sum()).reshape(3,3)
# print(np.array(CAD.isnull().sum()))
sns.heatmap(CAD.isnull(), cmap='viridis', cbar=False)
plt.title('nan Values')

In [ ]:
X=CAD_selected
# CAD_selected.head(1)
# Y=df_y=data["y"]

In [ ]:
col_names=X.columns
# print(col_names)
scaler=StandardScaler()
scaler.fit(X)
X[col_names]=scaler.transform(X)
CAD_selected['day']=CAD["date"].apply(get_day)
CAD_selected['hour']=CAD["date"].apply(get_hour)
# X.head(5)

In [ ]:
Y=data["y"]
# X.info()
# Y.info()
x_train,x_test,y_train,y_test=train_test_split(X,Y,test_size=0.20,random_state=101)
model=LinearRegression()
model.fit(x_train,y_train)
y_predict=model.predict(x_test)
# print(y_predict[:10])
# print(y_test[:10])
score=model.score(x_train,y_train)
# print(score)
# print(model.coef_)
# print(model.intercept_)
# MAE=mean_absolute_error(y_test,y_predict)
# print(MAE)

In [ ]:
coeff_df = pd.DataFrame(model.coef_,X.columns,columns=['Coefficient'])
coeff_df



In [ ]:
model2=SVR()
model2.fit(x_train,y_train)
y_predict2=model2.predict(x_test)
# print(y_predict2[:10])
# print(y_test[:10])
score2=model2.score(x_train,y_train)
# print(score2)
# MAE=mean_absolute_error(y_test,y_predict2)
# print(MAE)

In [ ]:
date_to_test={"CAD-open":-1.326500 ,"CAD-high":-1.33079,"CAD-low":-1.329604 ,"CAD-close":-1.329545 ,"day":3 ,"hour":9 }
df_test=pd.DataFrame([date_to_test])
# df_test
m1=model.predict(df_test)
print(m1)
m2=model2.predict(df_test)
print(m2)

In [ ]:
def save_pipeline(model, name_model, num_pipeline, num_model):

    pipeline= Pipeline([
        ('scaler', StandardScaler()),
        (name_model, model())
    ])

    # Fit the pipeline to the data
    pipeline.fit(x_train, y_train)

    # Save the first transformation pipeline
    joblib.dump(pipeline, f'pipeline{num_pipeline}.pkl')

    # Save the first model
    joblib.dump(pipeline.named_steps[name_model], f'model{num_model}.pkl')

save_pipeline(LinearRegression,"Linear_reg",1,1)
save_pipeline(SVR,"SVR_model",2,2)